# Open-Meteo data ingestion

Fetch hourly weather observations for Sydney from the Open-Meteo **Historical Weather
API** and **store them as a CSV file in `data/raw`**.

## Source

| Endpoint | Coverage | Notes |
|---|---|---|
| `https://archive-api.open-meteo.com/v1/archive` | 1940 onwards | The Historical Weather API named in the assignment brief. Reanalysis output (ERA5 0.25 deg from 1940, ERA5-Land 0.1 deg from 1950, ECMWF IFS 9 km from 2017). It serves every variable required by both target formulas, so this is the only endpoint used. |

## Variables requested

Only the variables named in the two target formulas are pulled. Nothing else is
requested: any additional predictor used later in the project is derived from these
seven series through lagging, rolling windows and calendar encoding, so no extra API
column is required.

| Variable | Climate Comfort Index | Weather Hazard Category |
|---|---|---|
| `temperature_2m` | `TempScore` | `TempHazard` |
| `relative_humidity_2m` | `HumidityScore` | - |
| `wind_speed_10m` | `WindScore` | - |
| `cloud_cover` | `CloudScore` | `CloudHazard` |
| `precipitation` | `RainScore` | `RainHazard` |
| `wind_gusts_10m` | - | `WindHazard` |
| `snowfall` | - | `SnowHazard` |

`temperature_2m`, `cloud_cover` and `precipitation` appear in both formulas, so the
request is the seven-element union of the two lists.

## Scope

Observations are requested from `2016-01-01` to `2025-12-31`. The assignment reserves
2026 onwards as unseen production data, so it is never downloaded here.

## Units returned by the API

Logged from `hourly_units` at fetch time and confirmed against the endpoint
documentation: `temperature_2m` °C, `relative_humidity_2m` %, `wind_speed_10m` and
`wind_gusts_10m` km/h, `cloud_cover` %, `precipitation` mm, **`snowfall` cm** (not mm).
These feed directly into the target formulas, so the snowfall unit in particular is
worth recording: `SnowHazard = MIN(snowfall / 15, 1)` is evaluated in centimetres.

In [1]:
from __future__ import annotations

import logging
import time
from dataclasses import dataclass
from pathlib import Path

import pandas as pd
import requests

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
logger = logging.getLogger(__name__)

## Configuration

Variables are declared per target so that the requirements of each model stay explicit
and traceable back to the brief. The request sent to the API is the union of the two
lists.

In [2]:
@dataclass(frozen=True)
class Location:
    """A geographic point together with the timezone used for aggregation."""

    latitude: float
    longitude: float
    timezone: str
    name: str


SYDNEY = Location(
    latitude=-33.8688,
    longitude=151.2093,
    timezone="Australia/Sydney",
    name="Sydney",
)

ARCHIVE_URL = "https://archive-api.open-meteo.com/v1/archive"

# Required by the Climate Comfort Index formula.
CCI_VARIABLES: tuple[str, ...] = (
    "temperature_2m",
    "relative_humidity_2m",
    "wind_speed_10m",
    "cloud_cover",
    "precipitation",
)

# Required by the Weather Hazard Index formula.
WHC_VARIABLES: tuple[str, ...] = (
    "precipitation",
    "wind_gusts_10m",
    "cloud_cover",
    "snowfall",
    "temperature_2m",
)

# dict.fromkeys preserves order while dropping duplicates: temperature_2m,
# cloud_cover and precipitation appear in both target lists.
HOURLY_VARIABLES: tuple[str, ...] = tuple(dict.fromkeys(CCI_VARIABLES + WHC_VARIABLES))

print(f"CCI      ({len(CCI_VARIABLES)}): {', '.join(CCI_VARIABLES)}")
print(f"WHC      ({len(WHC_VARIABLES)}): {', '.join(WHC_VARIABLES)}")
print(f"requested ({len(HOURLY_VARIABLES)}): {', '.join(HOURLY_VARIABLES)}")

CCI      (5): temperature_2m, relative_humidity_2m, wind_speed_10m, cloud_cover, precipitation
WHC      (5): precipitation, wind_gusts_10m, cloud_cover, snowfall, temperature_2m
requested (7): temperature_2m, relative_humidity_2m, wind_speed_10m, cloud_cover, precipitation, wind_gusts_10m, snowfall


## Retrieval

`fetch_hourly` wraps the HTTP call with retries and three integrity checks that the API
will not raise for on its own:

1. **Column completeness.** If the selected model does not carry a requested variable,
   the response simply omits the key rather than erroring. Without an explicit check
   this surfaces much later as a `KeyError` during feature engineering.
2. **Timestamp regularity.** Timestamps are naive local time because `timezone` is set,
   and Sydney observes daylight saving. Duplicated or skipped hours around the October
   and April transitions would otherwise pass through `pd.to_datetime` unnoticed.
3. **Units.** Logged from `hourly_units` so the assumptions behind the target formulas
   are verified against the response rather than assumed.

In [3]:
class IngestionError(RuntimeError):
    """Raised when the API cannot be reached or returns an unusable payload."""


def fetch_hourly(
    variables: tuple[str, ...] | list[str] = HOURLY_VARIABLES,
    start_date: str = "2016-01-01",
    end_date: str = "2025-12-31",
    location: Location = SYDNEY,
    max_retries: int = 4,
    timeout: int = 180,
) -> pd.DataFrame:
    """Retrieve hourly observations from the Historical Weather API.

    Parameters
    ----------
    variables:
        Open-Meteo hourly variable names to request. Defaults to the seven
        variables required by the two target formulas.
    start_date, end_date:
        Inclusive ISO-8601 bounds (YYYY-MM-DD).
    location:
        Coordinates and timezone. Defaults to Sydney.
    max_retries:
        Attempts before raising, with exponential backoff between them.
    timeout:
        Per-request timeout in seconds.

    Returns
    -------
    pandas.DataFrame
        Hourly observations indexed by local timestamp.

    Raises
    ------
    ValueError
        If the API rejects the request with a 400 (bad dates or unknown variable).
    IngestionError
        If the endpoint is unreachable after ``max_retries`` attempts, or the
        response is missing requested variables.
    """
    params: dict[str, object] = {
        "latitude": location.latitude,
        "longitude": location.longitude,
        "start_date": start_date,
        "end_date": end_date,
        "hourly": ",".join(variables),
        "timezone": location.timezone,
    }

    last_error: Exception | None = None
    for attempt in range(1, max_retries + 1):
        try:
            response = requests.get(ARCHIVE_URL, params=params, timeout=timeout)
            if response.status_code == 400:
                raise ValueError(response.json().get("reason", response.text))
            response.raise_for_status()
            payload = response.json()
            break
        except requests.RequestException as exc:
            last_error = exc
            if attempt == max_retries:
                raise IngestionError(
                    f"Could not retrieve data from the archive endpoint "
                    f"after {max_retries} attempts"
                ) from last_error
            wait = 2**attempt
            logger.warning(
                "Attempt %d/%d failed (%s); retrying in %ds",
                attempt,
                max_retries,
                exc,
                wait,
            )
            time.sleep(wait)

    if "hourly" not in payload:
        raise IngestionError("Response contained no 'hourly' block")

    frame = pd.DataFrame(payload["hourly"])

    # Check 1: every requested variable came back.
    missing = set(variables) - set(frame.columns)
    if missing:
        raise IngestionError(
            f"The archive endpoint did not return: {sorted(missing)}. "
            "The selected model may not carry these variables."
        )

    frame["time"] = pd.to_datetime(frame["time"])
    frame = frame.set_index("time").sort_index()

    # Check 2: the index is a regular hourly series.
    duplicates = int(frame.index.duplicated().sum())
    steps = frame.index.to_series().diff().dropna()
    irregular = int((steps != pd.Timedelta(hours=1)).sum())
    if duplicates or irregular:
        logger.warning(
            "%d duplicate timestamps, %d non-hourly steps (check DST transitions)",
            duplicates,
            irregular,
        )

    # Check 3: units match what the target formulas assume.
    logger.info("units: %s", payload.get("hourly_units", {}))

    logger.info(
        "Retrieved %d hourly rows (%s to %s)", len(frame), start_date, end_date
    )
    return frame

## Paths and date range

`REFRESH` guards against re-downloading on every kernel restart. Set it to `True` after
changing the requested variables or dates, otherwise the existing CSV is reused.

In [4]:
def project_root() -> Path:
    """Walk upwards from the working directory to the folder holding pyproject.toml."""
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    return here


RAW_DIR = project_root() / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

START_DATE = "2016-01-01"
END_DATE = "2025-12-31"  # 2026 onwards is reserved as production data.
REFRESH = True  # Set back to False once the CSV is written.

RAW_CSV = RAW_DIR / "sydney_hourly.csv"

print(f"Writing CSV to: {RAW_CSV}")

Writing CSV to: /Users/juanvargas/UTS/3rd Semester/AdvanceML/Assignment_2/36120-26SP-AT2-25945320-experiments/data/raw/sydney_hourly.csv


In [5]:
def write_hourly_csv(
    csv_path: Path,
    variables: tuple[str, ...] | list[str] = HOURLY_VARIABLES,
    start_date: str = START_DATE,
    end_date: str = END_DATE,
    refresh: bool = False,
) -> Path:
    """Download hourly observations and write them to a CSV in data/raw."""
    if csv_path.exists() and not refresh:
        logger.info("CSV already exists, skipping download: %s", csv_path)
        return csv_path

    frame = fetch_hourly(
        variables=variables,
        start_date=start_date,
        end_date=end_date,
    )
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(csv_path)
    logger.info("Wrote %d rows to %s", len(frame), csv_path)
    return csv_path


raw_path = write_hourly_csv(
    csv_path=RAW_CSV,
    variables=HOURLY_VARIABLES,
    start_date=START_DATE,
    end_date=END_DATE,
    refresh=REFRESH,
)

print(f"raw -> {raw_path}")

INFO units: {'time': 'iso8601', 'temperature_2m': '°C', 'relative_humidity_2m': '%', 'wind_speed_10m': 'km/h', 'cloud_cover': '%', 'precipitation': 'mm', 'wind_gusts_10m': 'km/h', 'snowfall': 'cm'}
INFO Retrieved 87672 hourly rows (2016-01-01 to 2025-12-31)
INFO Wrote 87672 rows to /Users/juanvargas/UTS/3rd Semester/AdvanceML/Assignment_2/36120-26SP-AT2-25945320-experiments/data/raw/sydney_hourly.csv


raw -> /Users/juanvargas/UTS/3rd Semester/AdvanceML/Assignment_2/36120-26SP-AT2-25945320-experiments/data/raw/sydney_hourly.csv


## Coverage audit

The row count alone is not evidence of a usable series: Open-Meteo returns a full time
index and pads unavailable periods with `null` rather than truncating the response. The
reanalysis archive also lags real time by roughly five days, so the tail of 2025 is
worth inspecting rather than assuming.

This cell establishes the usable window and the per-column completeness that the
training period is drawn from. Quote the result in the report.

In [6]:
def audit_coverage(path: Path) -> pd.DataFrame:
    """Report per-column completeness and the usable date window."""
    frame = pd.read_csv(path, index_col="time", parse_dates=True)

    populated = frame.dropna(how="all")

    print(f"rows requested : {len(frame):,}")
    print(f"rows with data : {len(populated):,}")
    print(f"usable window  : {populated.index.min()} to {populated.index.max()}")
    print(f"fully complete : {frame.notna().all(axis=1).sum():,} rows\n")

    coverage = (
        frame.notna()
        .mean()
        .mul(100)
        .round(2)
        .rename("coverage_pct")
        .sort_values()
        .to_frame()
    )
    print(coverage.to_string())
    return frame


raw_df = audit_coverage(raw_path)

rows requested : 87,672
rows with data : 87,672
usable window  : 2016-01-01 00:00:00 to 2025-12-31 23:00:00
fully complete : 87,672 rows

                      coverage_pct
temperature_2m               100.0
relative_humidity_2m         100.0
wind_speed_10m               100.0
cloud_cover                  100.0
precipitation                100.0
wind_gusts_10m               100.0
snowfall                     100.0


## Target-variable sanity checks

Two properties of the formulas are worth confirming before any modelling, because both
affect the distribution of the targets and both need a documented decision in the
report.

**Snowfall.** Sydney records no snow at sea level, so `SnowHazard` is expected to be
identically zero. That makes it a zero-variance term carrying 0.10 of the Weather Hazard
Index, which compresses the achievable WHI range and pushes the class distribution
towards the lower categories.

**Hazard component ranges.** Each hazard term is capped at 1 by `MIN(...)`, so the share
of observations that saturate the cap indicates how much of the raw signal the formula
discards. Precipitation above 30 mm and gusts above 100 km/h are both rare in Sydney,
which is the second reason to expect few Extreme Risk labels.

In [7]:
summary = raw_df[list(HOURLY_VARIABLES)].describe().round(2)
print(summary.to_string())

snowfall = raw_df["snowfall"]
print(f"\nsnowfall (cm): max={snowfall.max()}, non-zero hours={(snowfall > 0).sum()}")

# Share of hours saturating each capped hazard term, on the hourly series.
caps = {
    "precipitation > 30 mm": raw_df["precipitation"] > 30,
    "wind_gusts_10m > 100 km/h": raw_df["wind_gusts_10m"] > 100,
    "snowfall > 15 cm": raw_df["snowfall"] > 15,
    "|temperature_2m - 22| > 25 C": (raw_df["temperature_2m"] - 22).abs() > 25,
}
print("\nhours saturating each hazard cap")
for label, mask in caps.items():
    print(f"  {label:<30} {mask.mean():.4%} ({int(mask.sum())} hours)")

       temperature_2m  relative_humidity_2m  wind_speed_10m  cloud_cover  precipitation  wind_gusts_10m  snowfall
count        87672.00              87672.00        87672.00     87672.00       87672.00        87672.00   87672.0
mean            17.64                 73.04           12.93        48.93           0.12           27.70       0.0
std              5.30                 17.01            6.70        40.12           0.53           13.33       0.0
min              1.10                  9.00            0.00         0.00           0.00            1.40       0.0
25%             13.80                 62.00            7.70         5.00           0.00           17.30       0.0
50%             17.80                 75.00           11.80        45.00           0.00           25.60       0.0
75%             21.30                 87.00           17.10        96.00           0.00           36.40       0.0
max             42.00                100.00           48.80       100.00          19.70 

## Findings

Record the outcome of the two cells above here, then carry these into the report:

- **Usable window.** _(fill in from the coverage audit)_
- **Missing values.** Any column below 100% coverage, and the imputation decision taken.
- **Snowfall.** Confirmed zero-variance for Sydney; note the implication for the 0.10
  weight in the Weather Hazard Index and for the expected class balance.
- **Hazard saturation.** How often each capped term reaches 1, and what that implies for
  the number of High and Extreme Risk observations available for training.
- **DST handling.** Any duplicate or missing hours flagged during retrieval.

## Next steps

1. Aggregate the hourly observations to daily. The scaling constants in the brief imply
   daily aggregates (20 mm for `RainScore`, 30 mm for `RainHazard`, 100 km/h for gusts),
   so decide and document whether each index is computed hourly and then averaged, or
   computed once from daily aggregates. The two are not equivalent.
2. Construct the CCI and WHC targets from the daily frame, then shift them by +3 and +7
   days respectively.
3. Build lagged and rolling features from day `t` and earlier only. No feature may draw
   on observations from the target day, which is the leakage risk the brief warns about.